# AutoE2E on Bench2Drive: what it is given, what it returns, and why it swerves

> **Outputs are not committed: run the notebook to see them.** Every figure, table and number the text refers to comes from a run, and git keeps none of them: `tests/test_notebooks_are_bare.py` holds this file to that.
>
> 1. **Once:** `make setup`, then `uv pip install --python .venv/bin/python -e ".[analysis]"` for Jupyter and what the notebook draws with.
> 2. **The drives:** three Bench2Drive routes driven by AutoE2E with `record = "true"` **and** `record_inputs = "true"` under `[benchmarks.bench2drive]`, so each tick's whole request is on disk (`inputs/<tick>.npz`, ~2.5 MB a tick). With the autoe2e pairing of `docs/models/autoe2e.md` ("On Bench2Drive"): `paddock run --model autoe2e --bench bench2drive --scene 1711,2286,2373`. `PADDOCK_CONFIG` (or `./paddock.toml`) names the site config whose `{runs}` holds them.
> 3. **Run it:** run all on the repo's `.venv` kernel, or headless from the repo root: `.venv/bin/jupyter execute --output='{notebook_name}_all_executed' tools/models/autoe2e/analysis/autoe2e_bench2drive_deep_dive.ipynb`. That writes an executed copy beside this file, which git ignores. Never `--inplace`: it writes the outputs into this file, and `make check` fails on that. A run in Jupyter or VS Code saves them here too; strip it before committing, with `.venv/bin/python -m tools.strip_notebooks tools/models/autoe2e/analysis/autoe2e_bench2drive_deep_dive.ipynb`.
>
> **What it needs.** AutoE2E's environment and checkpoint (`paddock doctor --model autoe2e`) and a GPU: the network runs in AutoE2E's own interpreter, through [`autoe2e_bench2drive_probe.py`](autoe2e_bench2drive_probe.py) beside this file, never in this kernel. Its results are cached under `{runs}/bench2drive/_analysis/autoe2e/deep_dive/`, so only a route's first run costs model time (~10-20 minutes a route on an A100 shared with a sweep).

**Why.** In the 220-route sweep the same thing happens on route after route: AutoE2E launches hard from the standstill every Bench2Drive route starts in, reaches ~12 m/s in about 4 s, and then turns sharply off its lane -- usually to the left, often by 80-130 degrees -- into the scenery, or drives into whatever is in its lane. Most routes end 4-6 s in. This notebook takes three of those drives apart, tick by tick, to answer five questions:

1. **What exactly is the model given?** The six camera tiles, the eight-frame camera history and the geometry that ties them to the ground, the 64-row ego history, the route.
2. **What does it return, and how does that become a steering wheel and a pedal?**
3. **When and how does control pass between CARLA and the model**, and what in that handover could explain the behaviour?
4. **Does the input history drive the behaviour, and would an extrapolated past help?** Answered by changing one input of the real network call at a time and watching the plan.
5. **Does it see the cars it hits?** Answered from the model's own BEV segmentation head.

The adapter is not modified anywhere here: [`autoe2e_bench2drive_probe.py`](autoe2e_bench2drive_probe.py) replays each recorded tick into `AutoE2EPolicy.infer` exactly as the node received it (`tools/benchmarks/bench2drive/replay.py`), keeps the tensors `infer` hands the network, and calls the network again with one of them changed. Section 1 checks that the replay reproduces what happened in CARLA.

## Parameters

- **`ROUTES`**: the three drives, by Bench2Drive route id. Each needs a recorded drive with its inputs under `{runs}`.
- **`FOCUS_ROUTE`**: the route Section 3 takes apart input by input (the others get the same figures in Section 5).
- **`EXPERT`**: where `tools/benchmarks/bench2drive/expert.py` put Think2Drive's drives of the routes, or `None`.
- **`SERIES_EVERY`**: forward ticks between points of the along-the-drive counterfactuals (2 = every 0.2 s).
- **`RECOMPUTE`**: ignore the probe's cache and run the network again.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from paddock.config import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tools/models/autoe2e/analysis"))  # the lib, beside this notebook
import autoe2e_bench2drive_deep_dive_lib as L

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

ROUTES = ["1711", "2286", "2373"]
FOCUS_ROUTE = "2286"
EXPERT = None  # e.g. Path("/path/to/expert"); the routes are drawn against their route alone without it
SERIES_EVERY = 2
RECOMPUTE = False

In [ ]:
cfg = L.site()
routes = L.recorded_routes(cfg, ROUTES)
focus = {rid: r.focus_ticks() for rid, r in routes.items()}
series = {rid: L.run_probe(cfg, r, sorted(set(focus[rid].values())), every=SERIES_EVERY, recompute=RECOMPUTE) for rid, r in routes.items()}
pd.DataFrame([{**r.describe(), **{f"tick {k}": v for k, v in focus[rid].items()}} for rid, r in routes.items()]).set_index("route")

## 1 The three drives

One row per route above: how the leaderboard scored it, what ended it (the early stop's rule) and the three ticks taken apart below -- `settled` (1.5 s in, rolling and nothing yet wrong), `onset` (0.5 s before the car first leaves its lane line by a metre or turns 8 degrees off it) and `late` (just before the end). Each is a *forward* tick, one the node ran the network on.

The figures: the route (blue), the car's drive (black), the expert's (orange, if `EXPERT` is set), and the front camera at the three ticks.

In [ ]:
for rid, r in routes.items():
    expert = Path(EXPERT) if EXPERT else None
    L.overview(r, focus[rid], expert_root=expert)
    plt.show()

**Does the replay reproduce CARLA?** Every point of the along-the-drive series replays the node's request into a fresh `AutoE2EPolicy` and compares the six waypoints it returns with the six the node returned in CARLA (`ticks.jsonl`). If these are millimetres, every counterfactual below differs from what happened by the one input it changes and nothing else.

In [ ]:
pd.concat({rid: L.replay_fidelity(s) for rid, s in series.items()})

## 2 The handover between CARLA and the model

**One tick.** Bench2Drive runs CARLA in synchronous mode: the world advances 0.05 s, the sensors render, and nothing moves again until the agent returns a control. So the model's slowness costs wall-clock time, never reaction time -- a 2 s forward pass is, to the car, instantaneous. What can cost reaction time is what the adapter does between forward passes. The figure below is one tick, with this route's measured costs:

- The agent (`adapters/benchmarks/bench2drive/agent.py`) renders six 800x450 cameras, reads the car's pose and speed, cuts the route the leaderboard gave it (a point every ~50 m) to the stretch from 10 m behind to 200 m ahead, and POSTs all of it to the model's node.
- The node (`adapters/models/autoe2e/policy.py`) **pushes the frame into its history every tick**, and runs the network only when 0.1 s of game time has passed since it last did (`AUTOE2E_REPLAN_S = "0.1"`, the model's own 10 Hz). On the tick between, it answers with the last plan moved into the car's current position.
- The network returns 64 steps of (acceleration, curvature) at 0.1 s. The adapter integrates them from the current speed into a path and samples a waypoint every 0.5 s -- but the wire format sends only **six** (`protocol.N_WAYPOINTS`, set for neuroNCAP), so the agent sees **the first 3 s of a 6.4 s plan**.
- The agent's tracker (`adapters/benchmarks/bench2drive/pid.py`, TCP's dual PID) steers toward the waypoint about 4 m ahead and sets its target speed to the waypoints' mean spacing over 0.5 s. **The plan's acceleration reaches the car only through how far apart those six waypoints are.**

In [ ]:
L.tick_flow_diagram(routes[FOCUS_ROUTE])
plt.show()

**Tick by tick.** For each route: (1) which ticks ran the network, (2) the speed the plan asked for and the speed the car had, (3) the controls, (4) where the car is in its lane and which way the plan turns, and (5) **how much of the model's past is real** -- the camera history is seven frames at -3.5 .. -0.5 s, and before 3.5 s of driving some of them do not exist, so the oldest real frame stands in for them; the ego history is 64 rows at 0.1 s, and before 6.4 s of driving the rows older than the route are filled (`steady_before_start`: steady driving at the first measured speed -- 0 here, since every route starts parked).

The dotted lines are the moments that matter: when the camera history first holds no stand-in, when the car first leaves its lane line, and the end.

In [ ]:
for rid, r in routes.items():
    fig = L.handover_timeline(r, L.events(r))
    fig.suptitle(f"Route {rid} {r.scenario}", y=1.01)
    plt.show()

**What the handover could do to the model -- the hypotheses tested below.**

- **H1, the launch.** Every route starts parked, and the tracker launches the car as hard as the plan's waypoint spacing asks. So for its first seconds the model's ego history says *standing still, then accelerating hard* -- a past the KITScenes fine-tune rarely saw (its samples begin 6.4 s into a recording, mid-drive). If the network reads that past as *a launch into a turn* -- a car pulling out of a junction -- it would plan more acceleration and a turn. Tested in Section 5 by replacing the ego history and keeping everything else.
- **H2, the camera history filling in.** Until 3.5 s, some history frames are stand-ins; at ~3.5 s they become real. The swerves begin at ~4 s. If the temporal fusion misreads frames taken from a moving car (an error in how each history frame is tied to the current pose), the plan would change exactly then. Tested by giving the network a motionless history.
- **H3, the feedback of its own turning.** Once the car yaws a little, the ego history's yaw-rate and curvature rows say so, and a network trained to continue the motion it is in would turn more: a loop. Tested by zeroing those rows.
- **H4, the route it is given.** The route is a point every ~50 m; on a bend, the corridor drawn between them cuts the corner. Tested by drawing it from the 1 m route instead, and by removing it.
- **H5, the 3 s the tracker sees.** The tracker reads the plan's first 3 s, so a plan that is straight for 3 s and turns after looks straight, and a plan's acceleration shows up only as waypoint spacing. This one is about how the plan is *read*, not what the network computes; Section 4 shows both.

## 3 What the model is given, step by step

The route `FOCUS_ROUTE`, at its `onset` tick: half a second before the car first leaves its lane line. The other routes get the same figures in Section 5.

In [ ]:
r, tick = routes[FOCUS_ROUTE], focus[FOCUS_ROUTE]["onset"]
F = L.focus_data(cfg, r, tick)
print(f"route {FOCUS_ROUTE}, tick {tick}: speed {F['v0']:.2f} m/s, reference height {float(F['ref_height']):.1f} m, route raster drawn: {bool(F['route_valid'])}")

### 3.1 The cameras

Top row: the six frames CARLA rendered, 800x450 at 70 degrees (the back camera at 110), from the agent's six-camera rig. Middle row: what the network was given -- each frame squashed to 512x512 (bilinear, then a JPEG round trip at quality 90, as upstream packs KITScenes), slot 0 being the 1024x1024 front tile (bottom) downsampled. The squash stretches every view vertically by 1.78x; the network was trained on views packed the same way, so this is its normal input, but the camera *rig* is not the one it was trained on (KITScenes' and nuPlan's cameras sit at other heights, angles and fields of view).

In [ ]:
L.cameras(r, tick, F)
plt.show()

### 3.2 The camera history

Seven frames, 3.5 s to 0.5 s ago, oldest first, then now. Each is tied to the *current* car position through its own projection (where the car was when it took that frame), so the network can line the past up with the present. A frame older than the route itself does not exist; the oldest real frame stands in for it, with its own pose (marked REPEATED).

In [ ]:
L.history(r, tick, F)
plt.show()

### 3.3 The geometry

Left: where each camera looks, in the car's frame, over the scene as CARLA had it (actors as boxes: cyan vehicles, grey static props -- including the parked cars Bench2Drive spawns as static meshes -- magenta pedestrians; the route dashed). Right: how many cameras see each ground cell of the model's 450x300 grid, from the projections the network was given. Ground the network cannot see is ground it can only guess about.

In [ ]:
L.geometry(r, tick, F)
plt.show()

### 3.4 The ego history

64 rows, 6.4 s to 0.1 s ago: speed, acceleration, yaw rate and curvature, differenced from the poses the agent sent (not read from the speedometer). Shaded: rows older than the route, filled as steady driving at the first measured speed -- zero, since every route starts parked. The newest acceleration is masked to zero, as the KITScenes stage trained.

In [ ]:
L.ego_rows(r, tick, F)
plt.show()

### 3.5 The route

The only navigation input: a corridor 1.75 m either side of the route polyline the agent sent (the leaderboard's plan points, ~50 m apart), clipped 10 m behind, and a 2 m disk where the route ends if that is within 200 m. No map: the 14-channel semantic map is all zeros with `map_valid` false, upstream's own no-map track, because a Bench2Drive agent is given no map. Over it, the truth: the 1 m route (dashed) and the actors.

In [ ]:
L.route_raster(r, tick, F)
plt.show()

## 4 What it returns, and how that becomes a control

Left: the 64 steps of acceleration and curvature the network returned. Middle: the adapter's integration of them into a path (0.1 s steps, from the history's speed), over the truth; the six dots are the waypoints the agent actually receives, and the star is where the tracker aims, with the speed it targets. Right: the same waypoints on the front camera, and the control the agent sent CARLA that tick.

In [ ]:
L.outputs(r, tick, F)
plt.show()

### 4.1 Does it see the cars?

AutoE2E has a BEV segmentation head: 8 classes on the same 450x300 grid as the route, read from the camera BEV that every plan is computed from. The KITScenes fine-tune froze that camera BEV and kept the head from the nuPlan stage, so it shows what the frozen BEV encodes. Each map is a per-class probability (the head was trained with a per-class sigmoid). If the vehicles CARLA placed light up in `vehicle`, the network sees them and the planner does not act on them; if they do not, the planner never had them.

In [ ]:
L.segmentation(r, tick, F)
plt.show()
L.seen_actors(r, tick, F)

## 5 What moves the plan: one input changed at a time

At each focus tick, the real network call again with one input changed:

| variant | what changes |
|---|---|
| ego rows: no turning | the 64 ego rows with yaw rate and curvature zeroed: the past was straight (H3) |
| ego rows: steady straight past | all 64 rows steady driving at today's speed: no launch, no turn -- an extrapolated, even past (H1) |
| ego rows: pre-start = today's speed | only the rows older than the route, steady at today's speed instead of the start's 0 |
| ego rows: standing still | all 64 rows zero |
| camera history: no motion | the seven history frames all the current one, from the current pose (H2) |
| no route raster / route from the 1 m route | (H4) |
| reference 1.5 m up / on the ground | the projections built from another height above the ground (the KITScenes fine-tune used 2.1 m, its LiDAR's) |

The numbers: the plan's first curvature and its mean over the first second (left +), its lateral offset 1, 2 and 3 s ahead (left +), its heading at 3 s, its first acceleration and its speed 2 s ahead.

In [ ]:
for rid in ROUTES:
    for name, t in focus[rid].items():
        print(f"route {rid} {routes[rid].scenario}, {name} tick {t} (speed {series[rid]['focus'][str(t)]['v0']} m/s)")
        display(L.variants_table(series[rid], t))

In [ ]:
for rid in ROUTES:
    t = focus[rid]["onset"]
    L.outputs(routes[rid], t, L.focus_data(cfg, routes[rid], t), variants=("live", "ego_zero_turn", "ego_steady", "cam_static", "route_dense"))
    plt.show()

**Along the drive.** The same counterfactuals at every `SERIES_EVERY`-th forward tick from the start to the end: the plan's turn over its first second and its offset 2 s ahead, as fed and as each counterfactual makes it, above what the car did. Where the black line (as fed) and a coloured one part, that input is moving the plan.

In [ ]:
for rid, r in routes.items():
    fig = L.series_plot(r, series[rid], L.events(r))
    fig.suptitle(f"Route {rid} {r.scenario}", y=1.01)
    plt.show()

**The other two routes, input by input** (the figures of Sections 3 and 4 at their `onset` tick).

In [ ]:
for rid in ROUTES:
    if rid == FOCUS_ROUTE:
        continue
    t = focus[rid]["onset"]
    G = L.focus_data(cfg, routes[rid], t)
    for draw in (L.cameras, L.ego_rows, L.route_raster, L.segmentation):
        draw(routes[rid], t, G)
        plt.show()
    display(L.seen_actors(routes[rid], t, G))

## 6 Closed loop: would another past drive better?

Section 5 changes the plan *offline*, one tick at a time, along a drive that already went wrong. Whether a different past makes the car *drive* better is a closed-loop question: every plan moves the car, and the car's motion becomes the next tick's past. So the same three routes were driven again with part of the 64 ego rows replaced at every tick, and everything else as before (`AUTOE2E_EGO_HISTORY`, set as a pairing in `[models.autoe2e.benchmarks.bench2drive.env]`; such a run is `valid: false`, since it is not the model's own input). The rows split two ways: speed and acceleration, which on these routes carry the launch from standstill, and yaw rate and curvature, which carry the turning. The four histories are every combination of the two:

| `AUTOE2E_EGO_HISTORY` | speed, acceleration | yaw rate, curvature |
|---|---|---|
| `real` | as measured | as measured |
| `no_turn` | as measured | zero |
| `steady_speed` | steady at the current speed | as measured |
| `steady` | steady at the current speed | zero: with the row above, an extrapolated straight past |

All the drives sit under the same `{runs}`, told apart by each run's recorded `model_env`.

In [ ]:
by_mode = L.drives(cfg, ROUTES)
print({mode: sorted(rs) for mode, rs in by_mode.items()})
table = L.closed_loop_table(by_mode)
table

**Driving score by route and history**, the same numbers as above side by side (100 = route completed with no infraction):

In [ ]:
order = [m for m in ("real", "no_turn", "steady_speed", "steady") if m in by_mode]
table["DS"].unstack()[order].round(1)

In [ ]:
for rid in ROUTES:
    L.closed_loop_figure(by_mode, rid)
    plt.show()

## 7 What this shows, and what the adapter could add

Measured on these three drives, each replayed to within 0.5 mm of the plans the node sent in CARLA (Section 1); the sweep's other routes end the same way (most 4-6 s in, at ~12 m/s, turned 80-130 degrees off the lane into scenery), but they were not replayed.

**Verified offline (Sections 3-5)**

1. **The inputs are what they should be.** Every camera points where the rig places it and every ground cell of the model's grid is seen by at least one (3.3); the history frames are the ones 3.5 .. 0.5 s back, with stand-ins only before 3.5 s of driving (3.2); the route raster is drawn where the route is (3.5).
2. **The ego history, not the cameras, the route or the geometry, sets the plan's turn and its acceleration.** At every focus tick of all three routes, freezing the camera history, drawing the route from the 1 m route or moving the reference height changes the plan's offset 2 s ahead by centimetres; replacing the 64 ego rows changes it by metres (Section 5).
3. **The launch.** Every Bench2Drive route starts parked, so the measured past soon shows hard acceleration, and the network continues it: +2.0 to +3.4 m/s² at every focus tick, a target speed above the car's, so the tracker holds the throttle at its 0.75 cap (Section 2), and 11-12 m/s by 4 s on a street posted at 8.3 m/s. With a steady past at the same speed, the first acceleration is -0.5 to +0.1 m/s².
4. **The turn, and the loop that makes it a spin.** The plan starts to turn left around 2.5-3 s (1711, 2286) or 4.5 s (2373), at 6-9 m/s. Once the car yaws, its yaw-rate and curvature rows say so, and the next plan turns harder: the plan's curvature grows exponentially to 0.35-0.5 1/m, a 2-3 m radius. Zeroing those rows caps it near 0.01 1/m and turns the late plans straight again (2286 at 4.9 s: 0.317 1/m as fed, 0.0009 with the rows zeroed); a steady past keeps it near 0.003-0.005 throughout.
5. **Not the camera history filling in (H2):** the turns start before the history holds no stand-in, and a motionless history leaves every plan as it was.
6. **Not the route's coarseness (H4):** the 1 m route gives the same plans as the 50 m plan points, and no route at all turns slightly *more* to the left. The route raster pulls the plan only weakly.

**Verified in closed loop (Section 6): each route driven four times, only the ego rows changed**

| driving score | `real` | `no_turn` | `steady_speed` | `steady` |
|---|---|---|---|---|
| 1711 | 8.4 | 9.3 | 8.4 | **30.8** |
| 2286 | 16.7 | 25.2 | 16.7 | **92.3** |
| 2373 | 23.4 | 29.6 | 16.8 | **30.8** |
| mean | 16.1 | 21.4 | 14.0 | **51.3** |

7. **Two loops run through the ego history, and the measured past closes both.**
   - *Speed:* fed its measured speed and acceleration, the network keeps the launch going. With only the turning zeroed (`no_turn`) the car reaches 16.5 m/s on 2286 and 18.6 m/s on 2373, a street posted at 8.3 m/s, its plan still asking for +1.2 m/s² at 7 s: faster than with the whole measured past, whose spin cut the drive short.
   - *Turning:* fed its measured yaw rate and curvature, the plan turns harder in whichever direction it already leans. After a launch it leans left and spins (2286: 0.33 1/m, 87 degrees off the route). With a steady speed it leans right, and the same loop makes that a right turn off the road (`steady_speed`, 2286: -0.01 1/m at 3 s, -0.07 at 6 s, -0.23 at 7 s).

   Cutting one loop is not enough: `no_turn` still crashes at 4.0 and 5.8 s, `steady_speed` at 5.6, 7.1 and 13.4 s.
8. **Cutting both, with a steady past, drives best on every route:** a mean DS of 51.3 against 16.1, at 4-6 m/s throughout; 2286 completes.
9. **What a steady past does not fix:**
   - Nothing corrects the plan's slight right bend (-0.01 to -0.02 1/m, a 50-100 m radius): 1711 drifts 11 m off its route and into a building at 32 s, 99 % of the way along.
   - It does not take the turn at 2373's junction: it carries straight on and leaves the route at 17.7 s. With no turning in its past and a route raster that pulls weakly (6), nothing starts the turn.
   - Every one of the four runs the red light on 2373: at 3-4 m/s with a steady past as at 12 m/s with the measured one. That is not the history.

Distances and headings in Section 6 are against the route, not the lane: inside a junction the nearest lane can be any crossing lane.

**Not settled**

- **Whether it sees the cars.** The BEV segmentation head's maps are streaks along the camera rays, near 0.5 everywhere (4.1). Recorded vehicles score 0.37-0.85 inside their boxes with no pattern, the car straight ahead 0.37-0.50. Either this checkpoint's head means nothing (the KITScenes stage loaded it and never ran it), or the frozen camera BEV does not localise objects in CARLA's frames. One nuPlan or KITScenes frame through the same probe would tell which; this site has neither. That the plan keeps accelerating at a car in its own lane fits both readings.
- **Why a launch reads as a turn.** Argued, not shown: the KITScenes fine-tune's samples start 6.4 s into a recording, so a launch from standstill is rare in what it learned from, and in recorded driving a car accelerating hard from a stop is usually pulling out of a junction, often turning.
- **Whether three routes speak for 220.** A steady past is best on all three, by a wide margin on two; the full benchmark under it is the test.

**What the adapter could add, in the order the evidence supports**

1. **A steady past** (`AUTOE2E_EGO_HISTORY = "steady"`): of the four histories, the only one that cuts both loops, and the best on every route (Section 6). It also ends the over-speed, so a cap at the posted limit is no longer needed. It is not upstream's input: a run with it is `valid: false`, a choice of the adapter recorded as one.
2. **A heading correction that cannot run away.** What a steady past gives up is any sign that the car's heading is off. The measured turning rows cannot be that sign: they amplify whatever the plan leans towards (`steady_speed`). It would have to come from the route, which the raster as trained barely conveys, or from outside the model, at which point the model is no longer what drives. Untested.
3. **The rest of the plan to the tracker.** The wire contract sends every closed-loop benchmark six waypoints (3 s), because neuroNCAP needs six; Bench2Drive's tracker would read 4 s. A per-benchmark waypoint count would give it the plan AutoE2E computes.
4. **A map.** It is the input the route raster was trained beside: upstream only ever sets `route_valid` with `map_valid`, and on its own the route barely moves the plan. A Bench2Drive SENSORS agent is given no map, so this is a limit of the pairing rather than something the adapter can fix.